# 🔍 Notebook 09: Model Diagnostics Checklist — Is Your MMM Any Good?

You’ve fit a Marketing Mix Model and the numbers look promising. But can you trust the results?

This notebook walks through **every diagnostic you should check** after fitting an MMM, with clear pass/fail criteria for each. We cover:

1. **MCMC Convergence** — Did the sampler explore the posterior properly?
2. **Fit Quality** — MAPE, RMSE, R²
3. **Residual Analysis** — Are residuals random noise?
4. **Durbin-Watson** — First-order autocorrelation
5. **Ljung-Box** — Higher-order autocorrelation
6. **Contribution Sanity** — Does the decomposition make business sense?
7. **Posterior Predictive Check** — Does simulated data look like real data?
8. **Summary Scorecard** — One-page pass/fail

**This notebook fits a real PyMC-Marketing MMM** using MCMC sampling, so all diagnostics run on genuine posterior draws.

## Setup

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import arviz as az
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
import warnings
warnings.filterwarnings('ignore')

from pymc_marketing.mmm import MMM, GeometricAdstock, TanhSaturation

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': '#FAFBFC',
    'axes.facecolor': '#FAFBFC',
    'axes.edgecolor': '#D0D7DE',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.color': '#D0D7DE',
    'legend.framealpha': 0.9,
    'legend.edgecolor': '#D0D7DE',
})

COLORS = ['#2563EB', '#F97316', '#10B981', '#EF4444', '#8B5CF6', '#EC4899']

import os
os.makedirs('images', exist_ok=True)

print('Setup complete.')

Setup complete.


## Load Data and Fit a Real MMM

We load the sample weekly dataset and fit a **real PyMC-Marketing MMM** using MCMC sampling.
This gives us genuine posterior draws for all diagnostics that follow.

In [2]:
# Load data
df = pd.read_csv('data/sample_mmm_weekly.csv', parse_dates=['date'])

channels = ['tv_spend', 'facebook_spend', 'google_search_spend']

X = df[['date'] + channels]
y = df['revenue']

print(f'Loaded {len(df)} weeks of data: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Revenue range: ${y.min():,.0f} to ${y.max():,.0f}')
print(f'Channels: {channels}')

Loaded 104 weeks of data: 2023-01-02 to 2024-12-23
Revenue range: $388,468 to $517,165
Channels: ['tv_spend', 'facebook_spend', 'google_search_spend']


In [3]:
# Fit the MMM
mmm = MMM(
    date_column='date',
    channel_columns=channels,
    adstock=GeometricAdstock(l_max=8),
    saturation=TanhSaturation(),
    yearly_seasonality=2,
)

mmm.fit(
    X=X, y=y,
    chains=2, draws=500, tune=500,
    cores=1, target_accept=0.95,
    random_seed=42,
)

print('Model fit complete.')
print(f'Posterior shape: {dict(mmm.idata.posterior.dims)}')

Initializing NUTS using jitter+adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [intercept, adstock_alpha, saturation_b, saturation_c, gamma_fourier, y_sigma]


Output()

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 324 seconds.


...

We recommend running at least 4 chains for robust computation of convergence diagnostics


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Output()

Model fit complete.
Posterior shape: {'chain': 2, 'draw': 500, 'channel': 3, 'fourier_mode': 4, 'date': 104}


In [4]:
# Generate posterior predictive samples
mmm.sample_posterior_predictive(X, extend_idata=True)

print('Posterior predictive sampling complete.')

Sampling: [y]


Output()

Posterior predictive sampling complete.


In [5]:
# Extract predictions (posterior predictive mean)
pp = mmm.idata.posterior_predictive

# Get the data variable name from the posterior predictive group
pp_var = list(pp.data_vars)[0]
pp_draws_all = pp[pp_var].values  # shape: (chains, draws, n_obs)
pp_flat = pp_draws_all.reshape(-1, pp_draws_all.shape[-1])  # (chains*draws, n_obs)

predicted = pp_flat.mean(axis=0)
actual = y.values
dates = df['date'].values
n = len(actual)
residuals = actual - predicted

print(f'Predictions extracted: {n} observations')
print(f'Mean predicted: ${predicted.mean():,.0f}, Mean actual: ${actual.mean():,.0f}')

Predictions extracted: 104 observations
Mean predicted: $441,883, Mean actual: $441,919


---

## Diagnostic 1: MCMC Convergence

Before examining model fit, we must verify that the **MCMC sampler converged properly**.
If the chains haven't mixed well, all downstream results are unreliable.

Key checks:
- **R-hat < 1.01**: Chains have converged to the same distribution
- **ESS > 100**: Enough effective samples for reliable estimates
- **Energy plot**: No divergences or energy transition problems

In [6]:
# R-hat and ESS summary
summary = az.summary(mmm.idata, round_to=3)

# Check convergence criteria
rhat_max = summary['r_hat'].max()
ess_bulk_min = summary['ess_bulk'].min()
ess_tail_min = summary['ess_tail'].min()

rhat_pass = rhat_max < 1.01
ess_pass = ess_bulk_min > 100

print('=' * 60)
print('  DIAGNOSTIC 1: MCMC CONVERGENCE')
print('=' * 60)
print(f'  Max R-hat:        {rhat_max:.4f}  {"PASS" if rhat_pass else "FAIL"} (threshold: < 1.01)')
print(f'  Min ESS (bulk):   {ess_bulk_min:.0f}    {"PASS" if ess_pass else "FAIL"} (threshold: > 100)')
print(f'  Min ESS (tail):   {ess_tail_min:.0f}')
print()

# Show parameters with worst convergence
print('Parameters with highest R-hat:')
worst_rhat = summary.nlargest(5, 'r_hat')[['mean', 'sd', 'r_hat', 'ess_bulk']]
print(worst_rhat.to_string())
print()

convergence_pass = rhat_pass and ess_pass
print(f'Overall convergence: {"PASS" if convergence_pass else "INVESTIGATE"}')

  DIAGNOSTIC 1: MCMC CONVERGENCE
  Max R-hat:        1.0160  FAIL (threshold: < 1.01)
  Min ESS (bulk):   159    PASS (threshold: > 100)
  Min ESS (tail):   107

Parameters with highest R-hat:
                                                       mean        sd  r_hat  ess_bulk
mu[2023-12-25T00:00:00.000000000]                     0.876     0.010  1.016   852.330
y_original_scale[2023-12-25T00:00:00.000000000]  452821.996  5048.662  1.016   852.330
saturation_c[facebook_spend]                          1.319     0.790  1.014   158.758
mu[2023-05-29T00:00:00.000000000]                     0.907     0.014  1.014   300.656
y_original_scale[2023-05-29T00:00:00.000000000]  468913.860  7172.667  1.014   300.656

Overall convergence: INVESTIGATE


In [7]:
# Trace plots for key parameters
var_names = [v for v in mmm.idata.posterior.data_vars if mmm.idata.posterior[v].ndim <= 2]
var_names = var_names[:6]  # Limit to 6 for readability

axes_trace = az.plot_trace(mmm.idata, var_names=var_names, compact=True,
                           figsize=(14, max(3 * len(var_names), 6)))
plt.suptitle('Trace Plots (density + trace)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('images/09_trace_plots.png', dpi=180, bbox_inches='tight', facecolor='#FAFBFC')
plt.show()
print('Saved: images/09_trace_plots.png')

Saved: images/09_trace_plots.png


In [8]:
# Energy plot
fig, ax = plt.subplots(figsize=(10, 5))
az.plot_energy(mmm.idata, ax=ax)
ax.set_title('Energy Plot (BFMI Check)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/09_energy_plot.png', dpi=180, bbox_inches='tight', facecolor='#FAFBFC')
plt.show()
print('Saved: images/09_energy_plot.png')

Saved: images/09_energy_plot.png


![Trace Plots](./images/09_trace_plots.png)
![Energy Plot](./images/09_energy_plot.png)

**What to look for:**
- **Trace plots (right panels):** Chains should look like “hairy caterpillars” — well-mixed, overlapping, no trends or stuck regions.
- **Density plots (left panels):** Both chains should produce nearly identical distributions.
- **Energy plot:** The marginal energy and energy transition distributions should overlap substantially. Poor overlap signals sampling difficulties.

---

## Diagnostic 2: Fit Quality (MAPE, RMSE, R²)

The first thing to check: **how well does the model reproduce the observed revenue?**

We use three complementary metrics:

| Metric | Formula | Interpretation |
|--------|---------|---------------|
| **MAPE** | Mean Absolute Percentage Error | Average % miss per week |
| **RMSE** | Root Mean Squared Error | Typical miss in $ |
| **R²** | Coefficient of Determination | Variance explained (1.0 = perfect) |

**Pass criteria:** MAPE < 10%, R² > 0.8

In [9]:
# Calculate fit metrics
mape = np.mean(np.abs(residuals / actual)) * 100
rmse = np.sqrt(np.mean(residuals**2))
nrmse = rmse / actual.mean() * 100
ss_res = np.sum(residuals**2)
ss_tot = np.sum((actual - actual.mean())**2)
r_squared = 1 - ss_res / ss_tot

def status_emoji(val, good, acceptable, lower_is_better=True):
    if lower_is_better:
        if val <= good: return 'PASS'
        elif val <= acceptable: return 'ACCEPTABLE'
        return 'INVESTIGATE'
    else:
        if val >= good: return 'PASS'
        elif val >= acceptable: return 'ACCEPTABLE'
        return 'INVESTIGATE'

print('=' * 55)
print('  DIAGNOSTIC 2: FIT QUALITY')
print('=' * 55)
print(f'  MAPE:            {mape:.1f}%     {status_emoji(mape, 10, 15)}')
print(f'  RMSE:            ${rmse:,.0f}')
print(f'  Normalized RMSE: {nrmse:.1f}%')
print(f'  R-squared:       {r_squared:.3f}    {status_emoji(r_squared, 0.8, 0.6, lower_is_better=False)}')
print()
print(f'  The model explains {r_squared*100:.1f}% of revenue variance.')
print(f'  Average weekly miss: {mape:.1f}% (${rmse:,.0f})')

  DIAGNOSTIC 2: FIT QUALITY
  MAPE:            2.8%     PASS
  RMSE:            $17,360
  Normalized RMSE: 3.9%
  R-squared:       0.516    INVESTIGATE

  The model explains 51.6% of revenue variance.
  Average weekly miss: 2.8% ($17,360)


In [10]:
# Visualization: Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(dates, actual / 1000, color=COLORS[0], linewidth=1.5, label='Actual', alpha=0.9)
ax.plot(dates, predicted / 1000, color=COLORS[1], linewidth=1.5, label='Predicted', alpha=0.9)
ax.fill_between(dates, actual / 1000, predicted / 1000, alpha=0.1, color=COLORS[3])
ax.set_xlabel('Date')
ax.set_ylabel('Revenue ($K)')
ax.set_title('Actual vs Predicted Revenue')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}K'))

ax = axes[1]
ax.scatter(actual / 1000, predicted / 1000, color=COLORS[0], alpha=0.5, s=50, edgecolors='white')
min_val = min(actual.min(), predicted.min()) / 1000
max_val = max(actual.max(), predicted.max()) / 1000
ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Perfect fit')
ax.set_xlabel('Actual Revenue ($K)')
ax.set_ylabel('Predicted Revenue ($K)')
ax.set_title(f'Scatter (R² = {r_squared:.3f})')
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}K'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}K'))

plt.tight_layout()
plt.savefig('images/09_fit_quality.png', dpi=180, bbox_inches='tight', facecolor='#FAFBFC')
plt.show()
print('Saved: images/09_fit_quality.png')

Saved: images/09_fit_quality.png


![Fit Quality](./images/09_fit_quality.png)

**What to look for:**
- The time series overlay should track closely. Large persistent gaps indicate missing variables.
- The scatter plot should cluster tightly around the 45-degree line. Fan shapes indicate heteroscedasticity.
- A few outlier weeks (holidays, promotions) are normal and expected.

---

## Diagnostic 3: Residual Analysis

Good residuals are **random noise**. If you see patterns, the model is missing something.

We check four things:
1. **Residuals over time** — should look like white noise (no trends, no cycles)
2. **Residual distribution** — should be roughly normal and centered at zero
3. **QQ-plot** — quantiles should follow the diagonal
4. **Residuals vs fitted** — should show no fan shape (heteroscedasticity)

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.bar(range(n), residuals / 1000,
       color=[COLORS[0] if r >= 0 else COLORS[3] for r in residuals],
       alpha=0.7, width=1.0)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(2 * residuals.std() / 1000, color=COLORS[3], linestyle='--', alpha=0.5, label='±2σ')
ax.axhline(-2 * residuals.std() / 1000, color=COLORS[3], linestyle='--', alpha=0.5)
ax.set_xlabel('Week Index')
ax.set_ylabel('Residual ($K)')
ax.set_title('Residuals Over Time')
ax.legend()

ax = axes[0, 1]
ax.hist(residuals / 1000, bins=20, color=COLORS[0], alpha=0.7, edgecolor='white', density=True)
x_range = np.linspace(residuals.min() / 1000, residuals.max() / 1000, 100)
ax.plot(x_range, stats.norm.pdf(x_range, residuals.mean() / 1000, residuals.std() / 1000),
        color=COLORS[1], linewidth=2, label='Normal fit')
ax.set_xlabel('Residual ($K)')
ax.set_ylabel('Density')
ax.set_title('Residual Distribution')
ax.legend()

ax = axes[1, 0]
stats.probplot(residuals, dist='norm', plot=ax)
ax.set_title('Q-Q Plot')
ax.get_lines()[0].set_color(COLORS[0])
ax.get_lines()[0].set_alpha(0.6)
ax.get_lines()[1].set_color(COLORS[3])

ax = axes[1, 1]
ax.scatter(predicted / 1000, residuals / 1000, color=COLORS[0], alpha=0.5, s=50, edgecolors='white')
ax.axhline(0, color='black', linewidth=0.8)
z = np.polyfit(predicted, residuals, 2)
p_fit = np.poly1d(z)
x_sorted = np.sort(predicted)
ax.plot(x_sorted / 1000, p_fit(x_sorted) / 1000, color=COLORS[1], linewidth=2, alpha=0.7, label='Trend')
ax.set_xlabel('Fitted Value ($K)')
ax.set_ylabel('Residual ($K)')
ax.set_title('Residuals vs Fitted')
ax.legend()

plt.tight_layout()
plt.savefig('images/09_residual_analysis.png', dpi=180, bbox_inches='tight', facecolor='#FAFBFC')
plt.show()
print('Saved: images/09_residual_analysis.png')

Saved: images/09_residual_analysis.png


![Residual Analysis](./images/09_residual_analysis.png)

**Interpretation guide:**

| Panel | Good Sign | Bad Sign |
|-------|-----------|----------|
| Residuals over time | Random scatter around zero | Trends, cycles, or clusters of same-sign residuals |
| Distribution | Bell-shaped, centered at zero | Heavy tails, skewness, or bimodality |
| Q-Q plot | Points follow the diagonal | S-curves (heavy tails) or banana shapes (skew) |
| Residuals vs fitted | Flat band around zero | Fan shape (variance grows with fitted value) |

---

## Diagnostic 4: Durbin-Watson Test

The Durbin-Watson statistic tests for **first-order autocorrelation** in residuals.

$$DW = \frac{\sum_{t=2}^{T}(e_t - e_{t-1})^2}{\sum_{t=1}^{T} e_t^2}$$

| DW Value | Interpretation |
|----------|---------------|
| Close to 2.0 | No autocorrelation (PASS) |
| Close to 0.0 | Strong positive autocorrelation |
| Close to 4.0 | Strong negative autocorrelation |
| 1.5 – 2.5 | Acceptable range |

In [12]:
dw_stat = durbin_watson(residuals)

if 1.5 <= dw_stat <= 2.5:
    dw_status = 'PASS'
    dw_msg = 'No significant first-order autocorrelation.'
elif 1.0 <= dw_stat < 1.5 or 2.5 < dw_stat <= 3.0:
    dw_status = 'ACCEPTABLE'
    dw_msg = 'Mild autocorrelation detected. Consider adding lagged terms.'
else:
    dw_status = 'INVESTIGATE'
    dw_msg = 'Strong autocorrelation. Model is missing time-dependent structure.'

print('=' * 55)
print('  DIAGNOSTIC 4: DURBIN-WATSON TEST')
print('=' * 55)
print(f'  DW Statistic:  {dw_stat:.3f}')
print(f'  Status:        {dw_status}')
print(f'  {dw_msg}')
print()

  DIAGNOSTIC 4: DURBIN-WATSON TEST
  DW Statistic:  1.908
  Status:        PASS
  No significant first-order autocorrelation.



In [13]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(residuals[:-1] / 1000, residuals[1:] / 1000,
           color=COLORS[0], alpha=0.5, s=50, edgecolors='white')
ax.axhline(0, color='black', linewidth=0.5, alpha=0.5)
ax.axvline(0, color='black', linewidth=0.5, alpha=0.5)

slope, intercept = np.polyfit(residuals[:-1], residuals[1:], 1)
x_line = np.linspace(residuals.min(), residuals.max(), 100)
ax.plot(x_line / 1000, (slope * x_line + intercept) / 1000,
        color=COLORS[1], linewidth=2, alpha=0.7,
        label=f'Slope = {slope:.2f} (0 = no autocorrelation)')

ax.set_xlabel('Residual at t ($K)')
ax.set_ylabel('Residual at t+1 ($K)')
ax.set_title(f'Lag-1 Autocorrelation (DW = {dw_stat:.3f})')
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('images/09_durbin_watson.png', dpi=180, bbox_inches='tight', facecolor='#FAFBFC')
plt.show()
print('Saved: images/09_durbin_watson.png')

Saved: images/09_durbin_watson.png


![Durbin-Watson](./images/09_durbin_watson.png)

**Reading this plot:**
- If points cluster in the **top-right and bottom-left** quadrants: positive autocorrelation (DW < 2)
- If points cluster in the **top-left and bottom-right** quadrants: negative autocorrelation (DW > 2)
- If points scatter evenly across all quadrants: no autocorrelation (DW ~ 2)

---

## Diagnostic 5: Ljung-Box Test

The Ljung-Box test is a **more powerful** alternative to Durbin-Watson. Instead of testing only lag 1, it tests whether **any of the first K autocorrelations** are significantly different from zero.

$$Q = T(T+2) \sum_{k=1}^{K} \frac{\hat{\rho}_k^2}{T-k}$$

**Decision rule:** If p-value < 0.05 at any lag, autocorrelation is present at that lag.

In [14]:
lb_result = acorr_ljungbox(residuals, lags=10, return_df=True)

print('=' * 55)
print('  DIAGNOSTIC 5: LJUNG-BOX TEST')
print('=' * 55)
print(f'{"Lag":>5}  {"LB Statistic":>14}  {"p-value":>10}  {"Status":>12}')
print('-' * 55)
any_fail = False
for lag in lb_result.index:
    stat_val = lb_result.loc[lag, 'lb_stat']
    pval = lb_result.loc[lag, 'lb_pvalue']
    status = 'PASS' if pval >= 0.05 else 'FAIL'
    if pval < 0.05:
        any_fail = True
    print(f'{lag:>5}  {stat_val:>14.3f}  {pval:>10.4f}  {status:>12}')

print()
lb_status = 'INVESTIGATE' if any_fail else 'PASS'
print(f'Overall: {lb_status}')

  DIAGNOSTIC 5: LJUNG-BOX TEST
  Lag    LB Statistic     p-value        Status
-------------------------------------------------------
    1           0.039      0.8439          PASS
    2           0.263      0.8767          PASS
    3           0.351      0.9501          PASS
    4           0.896      0.9252          PASS
    5           2.753      0.7380          PASS
    6           4.994      0.5446          PASS
    7           5.141      0.6427          PASS
    8           6.475      0.5941          PASS
    9           6.744      0.6637          PASS
   10           7.682      0.6599          PASS

Overall: PASS


In [15]:
fig, ax = plt.subplots(figsize=(10, 5))
lags = lb_result.index
p_vals = lb_result['lb_pvalue'].values

bar_colors = [COLORS[2] if p >= 0.05 else COLORS[3] for p in p_vals]
ax.bar(lags, p_vals, color=bar_colors, alpha=0.8, edgecolor='white', width=0.7)
ax.axhline(0.05, color=COLORS[3], linestyle='--', linewidth=2, alpha=0.7,
           label='α = 0.05 threshold')
ax.set_xlabel('Lag')
ax.set_ylabel('p-value')
ax.set_title('Ljung-Box Test p-values by Lag')
ax.set_xticks(lags)
ax.legend()
ax.set_ylim(0, 1.05)

for i, (lag, p) in enumerate(zip(lags, p_vals)):
    ax.text(lag, p + 0.02, f'{p:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('images/09_ljung_box.png', dpi=180, bbox_inches='tight', facecolor='#FAFBFC')
plt.show()
print('Saved: images/09_ljung_box.png')

Saved: images/09_ljung_box.png


![Ljung-Box](./images/09_ljung_box.png)

**Interpretation:**
- Green bars (p >= 0.05): No significant autocorrelation at that lag. Good.
- Red bars (p < 0.05): Significant autocorrelation detected. The model may be missing a systematic pattern.

**Common fixes for failed autocorrelation tests:**
1. Add **trend** component (linear or GP-smoothed)
2. Add **seasonality** (Fourier terms)
3. Check for **missing control variables** (weather, holidays, competitor activity)

---

## Diagnostic 6: Contribution Sanity Checks

Even if the model fits well statistically, the **channel decomposition must make business sense**. Check:

1. **All contributions positive?** With positive priors on media coefficients, they should be.
2. **Contribution shares vs spend shares** — a channel spending 5% of budget should not account for 50% of incremental revenue.
3. **Base revenue share** — typically 40–70% for established brands.

In [16]:
# Extract channel contributions from the posterior
mean_contributions = mmm.compute_mean_contributions_over_time(original_scale=True)

channel_names = channels
total_revenue = actual.sum()

print('=' * 70)
print('  DIAGNOSTIC 6: CONTRIBUTION SANITY CHECKS')
print('=' * 70)

contrib_shares = {}
spend_shares = {}
total_spend = sum(df[ch].sum() for ch in channel_names)

print(f'{"Channel":<25} {"Contrib Share":>14} {"Spend Share":>12} {"Ratio":>8}')
print('-' * 70)

for ch in channel_names:
    contrib = mean_contributions[ch].values
    c_share = contrib.sum() / total_revenue * 100
    s_share = df[ch].sum() / total_spend * 100
    ratio = c_share / s_share if s_share > 0 else 0
    contrib_shares[ch] = c_share
    spend_shares[ch] = s_share
    flag = '' if 0.3 <= ratio <= 3.0 else ' <-- CHECK'
    print(f'{ch:<25} {c_share:>13.1f}% {s_share:>11.1f}% {ratio:>7.2f}x{flag}')

total_channel_contrib = sum(mean_contributions[ch].values.sum() for ch in channel_names)
base_share = (1 - total_channel_contrib / total_revenue) * 100
print(f'{"Base (intercept+other)":<25} {base_share:>13.1f}%')
print()

base_status = 'PASS' if 30 <= base_share <= 80 else ('ACCEPTABLE' if 20 <= base_share <= 90 else 'INVESTIGATE')
print(f'Base share: {base_share:.1f}% -- {base_status} (expected: 40-70%)')

  DIAGNOSTIC 6: CONTRIBUTION SANITY CHECKS
Channel                    Contrib Share  Spend Share    Ratio
----------------------------------------------------------------------
tv_spend                           15.0%        47.8%    0.31x
facebook_spend                     22.6%        23.2%    0.97x
google_search_spend                26.9%        29.0%    0.93x
Base (intercept+other)             35.5%

Base share: 35.5% -- PASS (expected: 40-70%)


In [17]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [2, 1]})

ax = axes[0]
channel_contribs = {}
for ch in channel_names:
    channel_contribs[ch] = mean_contributions[ch].values

total_channel = sum(channel_contribs[ch] for ch in channel_names)
base_contrib = actual - total_channel

stack_data = [base_contrib / 1000]
stack_labels = ['Base']
stack_colors = ['#94A3B8']

for i, ch in enumerate(channel_names):
    stack_data.append(channel_contribs[ch] / 1000)
    stack_labels.append(ch.replace('_spend', '').replace('_', ' ').title())
    stack_colors.append(COLORS[i % len(COLORS)])

ax.stackplot(dates, *stack_data, labels=stack_labels, colors=stack_colors, alpha=0.8)
ax.plot(dates, actual / 1000, color='black', linewidth=1.5, alpha=0.7, label='Actual Revenue')
ax.set_ylabel('Revenue ($K)')
ax.set_title('Revenue Decomposition by Channel')
ax.legend(loc='upper left', ncol=3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}K'))

ax = axes[1]
x_pos = np.arange(len(channel_names))
width = 0.35
c_vals = [contrib_shares[ch] for ch in channel_names]
s_vals = [spend_shares[ch] for ch in channel_names]
ax.bar(x_pos - width/2, c_vals, width, label='Revenue Contribution %', color=COLORS[0], alpha=0.8)
ax.bar(x_pos + width/2, s_vals, width, label='Spend Share %', color=COLORS[1], alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels([ch.replace('_spend', '').replace('_', ' ').title() for ch in channel_names])
ax.set_ylabel('Share (%)')
ax.set_title('Contribution Share vs Spend Share')
ax.legend()

plt.tight_layout()
plt.savefig('images/09_contribution_sanity.png', dpi=180, bbox_inches='tight', facecolor='#FAFBFC')
plt.show()
print('Saved: images/09_contribution_sanity.png')

Saved: images/09_contribution_sanity.png


![Contribution Sanity](./images/09_contribution_sanity.png)

**Rules of thumb:**
- **Contribution/Spend ratio between 0.3x and 3.0x** is reasonable.
- **Base revenue 40–70%** for an established brand is typical.
- If a low-spend channel shows outsized contribution, check for data leakage or confounding.

---

## Diagnostic 7: Posterior Predictive Check

A posterior predictive check asks: **if I simulate new data from the model, does it look like the real data?**

In a Bayesian MMM, this means drawing many samples from the posterior and checking that the observed revenue falls within the predicted interval (94% HDI).

In [18]:
# ArviZ posterior predictive check plot
fig, ax = plt.subplots(figsize=(14, 6))
az.plot_ppc(mmm.idata, num_pp_samples=100, ax=ax, kind='kde')
ax.set_title('Posterior Predictive Check (KDE)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/09_ppc_kde.png', dpi=180, bbox_inches='tight', facecolor='#FAFBFC')
plt.show()
print('Saved: images/09_ppc_kde.png')

Saved: images/09_ppc_kde.png


In [19]:
pp_lower = np.percentile(pp_flat, 3, axis=0)
pp_upper = np.percentile(pp_flat, 97, axis=0)
pp_mean = pp_flat.mean(axis=0)

outside = (actual < pp_lower) | (actual > pp_upper)
coverage = (1 - outside.mean()) * 100

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(dates, pp_lower / 1000, pp_upper / 1000,
                alpha=0.25, color=COLORS[0], label='94% HDI (3%-97%)')
ax.plot(dates, pp_mean / 1000, color=COLORS[0], linewidth=1.5, alpha=0.7, label='Posterior mean')
ax.plot(dates, actual / 1000, color='black', linewidth=1.5, alpha=0.8, label='Observed')

if outside.any():
    ax.scatter(dates[outside], actual[outside] / 1000,
               color=COLORS[3], s=60, zorder=5,
               label=f'Outside HDI ({outside.sum()} pts)', edgecolors='white')

ax.set_xlabel('Date')
ax.set_ylabel('Revenue ($K)')
ax.set_title(f'Posterior Predictive Check — Coverage: {coverage:.1f}%')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}K'))

plt.tight_layout()
plt.savefig('images/09_posterior_predictive.png', dpi=180, bbox_inches='tight', facecolor='#FAFBFC')
plt.show()

print(f'Coverage: {coverage:.1f}% of observations within 94% HDI')
print(f'Points outside HDI: {outside.sum()} of {n}')
print('Saved: images/09_posterior_predictive.png')

Coverage: 96.2% of observations within 94% HDI
Points outside HDI: 4 of 104
Saved: images/09_posterior_predictive.png


![PPC KDE](./images/09_ppc_kde.png)
![Posterior Predictive](./images/09_posterior_predictive.png)

**What to look for:**
- Most observations (> 85%) should fall within the 94% HDI band.
- Points outside the band are expected (roughly 6% of the time for a well-calibrated model).
- If many points cluster outside on one side, the model has a systematic bias.
- The KDE check should show observed data distribution overlapping with posterior predictive samples.

---

## Summary Scorecard

Here is the final pass/fail summary for all diagnostics.

In [20]:
def grade(val, good_fn, acceptable_fn):
    if good_fn(val):
        return 'PASS'
    elif acceptable_fn(val):
        return 'ACCEPTABLE'
    return 'INVESTIGATE'

lb_overall = 'PASS' if not any_fail else 'INVESTIGATE'
coverage_status = grade(coverage, lambda x: x >= 85, lambda x: x >= 70)

scorecard = [
    ('MCMC Convergence', f'R-hat={rhat_max:.3f}, ESS={ess_bulk_min:.0f}', '',
     'PASS' if convergence_pass else 'INVESTIGATE'),
    ('Fit Quality (MAPE)', f'{mape:.1f}%', '<10%', status_emoji(mape, 10, 15)),
    ('Fit Quality (R²)', f'{r_squared:.3f}', '>0.80',
     status_emoji(r_squared, 0.8, 0.6, lower_is_better=False)),
    ('Durbin-Watson', f'{dw_stat:.3f}', '1.5-2.5', dw_status),
    ('Ljung-Box', f'Any lag p<0.05: {any_fail}', 'All p>0.05', lb_overall),
    ('Base Revenue Share', f'{base_share:.1f}%', '40-70%', base_status),
    ('PPC Coverage (94% HDI)', f'{coverage:.1f}%', '>85%', coverage_status),
]

print('=' * 75)
print('  FINAL DIAGNOSTIC SCORECARD')
print('=' * 75)
print(f'{"Diagnostic":<25} {"Value":>20} {"Threshold":>12} {"Status":>12}')
print('-' * 75)
for name, val, threshold, status in scorecard:
    marker = '[OK]' if status == 'PASS' else ('[~~]' if status == 'ACCEPTABLE' else '[!!]')
    print(f'{marker} {name:<23} {val:>20} {threshold:>12} {status:>12}')

pass_count = sum(1 for s in scorecard if s[3] == 'PASS')
total_count = len(scorecard)
print()
print(f'Score: {pass_count}/{total_count} passed')
if pass_count == total_count:
    print('All diagnostics passed. Model is ready for business decisions.')
elif pass_count >= total_count - 2:
    print('Most diagnostics passed. Review the flagged items before using results.')
else:
    print('Multiple diagnostics flagged. Investigate before trusting model outputs.')

  FINAL DIAGNOSTIC SCORECARD
Diagnostic                               Value    Threshold       Status
---------------------------------------------------------------------------
[!!] MCMC Convergence        R-hat=1.016, ESS=159               INVESTIGATE
[OK] Fit Quality (MAPE)                      2.8%         <10%         PASS
[!!] Fit Quality (R²)                       0.516        >0.80  INVESTIGATE
[OK] Durbin-Watson                          1.908      1.5-2.5         PASS
[OK] Ljung-Box               Any lag p<0.05: False   All p>0.05         PASS
[OK] Base Revenue Share                     35.5%       40-70%         PASS
[OK] PPC Coverage (94% HDI)                 96.2%         >85%         PASS

Score: 5/7 passed
Most diagnostics passed. Review the flagged items before using results.


In [21]:
fig, ax = plt.subplots(figsize=(10, 5))

labels = [s[0] for s in scorecard]
statuses = [s[3] for s in scorecard]
colors_map = {'PASS': COLORS[2], 'ACCEPTABLE': '#F59E0B', 'INVESTIGATE': COLORS[3]}
bar_colors = [colors_map[s] for s in statuses]

y_pos = np.arange(len(labels))
ax.barh(y_pos, [1] * len(labels), color=bar_colors, alpha=0.8, height=0.6, edgecolor='white')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlim(0, 1.5)
ax.set_xticks([])
ax.set_title(f'Model Diagnostic Scorecard ({pass_count}/{total_count} Passed)')
ax.invert_yaxis()

for i, (status, val) in enumerate(zip(statuses, [s[1] for s in scorecard])):
    ax.text(1.05, i, f'{status}  |  {val}', va='center', fontsize=10)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLORS[2], alpha=0.8, label='PASS'),
    Patch(facecolor='#F59E0B', alpha=0.8, label='ACCEPTABLE'),
    Patch(facecolor=COLORS[3], alpha=0.8, label='INVESTIGATE'),
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('images/09_scorecard.png', dpi=180, bbox_inches='tight', facecolor='#FAFBFC')
plt.show()
print('Saved: images/09_scorecard.png')

Saved: images/09_scorecard.png


![Scorecard](./images/09_scorecard.png)

---

## What To Do When Diagnostics Fail

| Problem | Likely Cause | Fix |
|---------|-------------|-----|
| High MAPE / Low R² | Missing variables, wrong functional form | Add controls (weather, events), check adstock/saturation specs |
| Residual autocorrelation | Missing time-dependent structure | Add trend, seasonality, or AR component |
| Non-normal residuals | Outliers or structural breaks | Check for data issues, consider robust likelihood |
| Implausible contributions | Prior misspecification, collinearity | Tighten priors, check VIF, add lift test calibration |
| Poor PPC coverage | Model is overconfident or biased | Widen priors, add more flexibility (e.g., time-varying coefficients) |
| MCMC convergence issues | Complex posterior, poor parameterization | Increase tune/draws, use target_accept=0.95, reparameterize |

---

## Next Steps

- [Notebook 10: Lift Test Calibration](./10-lift-test-calibration.ipynb) — Validate with incrementality experiments
- [Notebook 11: Scenario Planning](./11-scenario-planning.ipynb) — Use the validated model for what-if analysis
- [Notebook 08: Tanh Saturation Deep Dive](./08-tanh-saturation-deep-dive.ipynb) — Understanding the saturation function